In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# RLHF Fine-Tuning

## Load the SFT and Reward Models

In [ ]:
%cp /content/drive/MyDrive/copy\ files/reward_model.pt .
%cp /content/drive/MyDrive/copy\ files/sft_model_epoch_1.zip .

In [ ]:
!unzip sft_model_epoch_1.zip

Archive:  sft_model_epoch_1.zip
   creating: sft_model_epoch_1/
  inflating: sft_model_epoch_1/generation_config.json  
  inflating: sft_model_epoch_1/config.json  
  inflating: sft_model_epoch_1/model.safetensors  


## Reward model

In [ ]:
import torch
from typing import Optional
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class RewardHead(nn.Module):
    """
    The RewardHead class implements a head for GPT2
    that returns a scalar for each output token.
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(self.reward.weight, std=(1.0 / np.sqrt(self.hidden_size + 1)))
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.reward(output)


class GPT2RewardModel(nn.Module):
    """
    GPT2 model with a reward head on top.
    """

    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        # config = self.llm.config
        # Add the reward head
        self.reward_head = RewardHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:

        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )

        # Get the last hidden state
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # Apply the reward head
        rewards = self.reward_head(last_hidden_state).squeeze(-1)

        return torch.sigmoid(rewards)



In [ ]:
model_name = "gpt2"
reward_model = GPT2RewardModel(model_name)
reward_model.load_state_dict(torch.load("reward_model.pt", map_location='cpu'))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

<All keys matched successfully>

## Model with Value Head

In [ ]:
import torch
from typing import Optional
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class ValueHead(nn.Module):
    """
    The ValueHead class implements a head for GPT2
    that returns a scalar for each output token.
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.value = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(self.value.weight, std=(1.0 / np.sqrt(self.hidden_size + 1)))
        nn.init.zeros_(self.value.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.value(output)


class ModelForCausalLMWithValueHead(nn.Module):
    """
    GPT2 model with a value head on top.
    """

    def __init__(self, model_path):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_path)
        # config = self.llm.config
        # Add the reward head
        self.v_head = ValueHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:

        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )
        lm_logits = transformer_outputs.logits
        # Get the last hidden state
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # Apply the reward head
        value = self.v_head(last_hidden_state).squeeze(-1)
        return lm_logits, value

    def generate(self, *args, **kwargs):
        return self.llm.generate(*args, **kwargs)


In [ ]:
model_path = './sft_model_epoch_1'
model = ModelForCausalLMWithValueHead(model_path)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

## Preparing Dataset

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
%pip install datasets==3.5.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 25.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.


In [ ]:
from datasets import load_dataset
dataset = load_dataset("sst2")
dataset

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [ ]:
ds_train, ds_val = dataset['train'], dataset['validation']

## Filtering

In [ ]:
len(ds_train)

67349

In [ ]:
ds_train = ds_train.filter(lambda x: len(x['sentence'].split(' ')) > 8)

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

In [ ]:
len(ds_train)

31105

In [ ]:
ds_val = ds_val.filter(lambda x: len(x['sentence'].split(' ')) > 8)

Filter:   0%|          | 0/872 [00:00<?, ? examples/s]

In [ ]:
len(ds_val)

807

In [ ]:
import random
input_min_token_length = 2
input_max_token_length = 8
input_token_length_range = list(range(input_min_token_length, input_max_token_length))
print(input_token_length_range)

[2, 3, 4, 5, 6, 7]


In [ ]:
random.choice(input_token_length_range)

3

In [ ]:
def tokenize(sample):
    input_size = random.choice(input_token_length_range)
    sample['input_ids'] = tokenizer.encode(sample['sentence'])[:input_size]
    sample['attention_mask'] = [1] * len(sample['input_ids'])
    sample['query'] = tokenizer.decode(sample['input_ids'])
    return sample

map_kwargs = {
    "batched": False,
    "remove_columns": ['idx', 'sentence', 'label']
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)


Map:   0%|          | 0/31105 [00:00<?, ? examples/s]

Map:   0%|          | 0/807 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')

In [ ]:
tokenized_dataset_train[6]

{'input_ids': tensor([ 1640,   883,  3807, 31006,   508, 13121]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1]),
 'query': 'for those moviegoers who complain'}

In [ ]:
REWARD_TOKEN_ID = tokenizer.eos_token_id

In [ ]:
from torch.utils.data import DataLoader

batch_size = 32

def collator(batch):
    return dict((key, [d[key] for d in batch]) for key in batch[0])

train_dataloader = DataLoader(tokenized_dataset_train, batch_size=batch_size, collate_fn=collator, shuffle=True)
val_dataloader = DataLoader(tokenized_dataset_val, batch_size=batch_size, collate_fn=collator, shuffle=True)

In [ ]:
batch = next(iter(train_dataloader))
batch

{'input_ids': [tensor([14150,   925,   257, 11004,  2646,   326, 25409]),
  tensor([5562,  373,  299,  470,  477,  326, 1049]),
  tensor([ 292,  262, 5337,  319,  543,  340,  705]),
  tensor([20797,   262]),
  tensor([  11, 1748]),
  tensor([39563,  4420]),
  tensor([5998,   78,  290,  474,  377]),
  tensor([24571,  3397,   561,   466,   880,   284, 33070]),
  tensor([8117,  389,  617, 6918,  326, 2277,  345]),
  tensor([ 9536,  1326,  3443, 31137,   287, 35197]),
  tensor([  82, 7126,  354, 7867]),
  tensor([4246,  501,  546,  644]),
  tensor([271, 326, 673]),
  tensor([  64,  717,   12, 4871]),
  tensor([11407,  1165,   537]),
  tensor([ 74, 280,  88, 378]),
  tensor([38171,   284, 26633,  5176]),
  tensor([ 260,  570,  286, 2046]),
  tensor([9062,  591,  484,  466,  299,  470, 1833]),
  tensor([10197,   287,   262]),
  tensor([338, 407]),
  tensor([ 1169, 22279,  7596, 23895, 27329]),
  tensor([   81, 14229,    12, 13288, 13990]),
  tensor([ 5661,   326,  7584,   781, 48295,   277, 

In [ ]:
output_min_length = 5
output_max_length = 16

# https://huggingface.co/docs/trl/how_to_train#how-to-generate-text-for-training

generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.pad_token_id
}

## Sample Generation

In [ ]:
new_tokens = random.choice(list(range(output_min_length, output_max_length)))
generation_kwargs["max_new_tokens"] = new_tokens
sample = tokenizer('Hi, this')
sample

{'input_ids': [17250, 11, 428], 'attention_mask': [1, 1, 1]}

In [ ]:
query_response = model.generate(
    input_ids=torch.tensor(sample['input_ids']).unsqueeze(0),
    attention_mask=torch.tensor(sample['attention_mask']).unsqueeze(0),
    **generation_kwargs
    ).squeeze(0)
query_response

tensor([17250,    11,   428,  3952,   318,  1223,   655,   329,  6490])

In [ ]:
tokenizer.decode(query_response)

'Hi, this park is something just for adults'

In [ ]:
with torch.no_grad():
    query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID])])
    attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
    score = reward_model(query_response_score.unsqueeze(0), attention_mask.unsqueeze(0)).squeeze(0)[-1]
score

tensor(0.6943)

## Batch Generation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
reward_model = reward_model.to(device)

query_tensors = batch['input_ids']
query_attention_masks = batch['attention_mask']

response_tensors = []
query_response_tensors = []
score_tensors = []

for i, query in enumerate(query_tensors):
    query = query.to(device)
    query_attention_mask = query_attention_masks[i].to(device)
    new_tokens = random.choice(list(range(output_min_length, output_max_length)))
    generation_kwargs["max_new_tokens"] = new_tokens
    query_response = model.generate(
        input_ids=query.unsqueeze(0),
        attention_mask=query_attention_mask.unsqueeze(0),
        **generation_kwargs
    ).squeeze(0)

    response_len = len(query_response) - len(query)
    response_tensors.append(query_response[-response_len:])
    query_response_tensors.append(query_response)

    with torch.no_grad():
        query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID]).to(device)])
        attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
        score = reward_model(query_response_score.unsqueeze(0), attention_mask.unsqueeze(0)).squeeze(0)[-1]
        score = 2 * (score - 0.5)
    score_tensors.append(score)

batch["response"] = [tokenizer.decode(response) for response in response_tensors]
print(batch['response'])

[' and appreciates a similar both surreal heartbreak and an', ' , but it delivered an update', 's based retains vivid wicked', ' page with no seriously pretentious editing', ' fathers are conflicted about what a film needs , asking', ' timely .    ', 'ian , to sum that up , almost', ' you together this summer \xa0', ' believeingly close to the edge of', ' you by insisting that to marry your best', ' a supremely effective pires \xa0directed by', ' you see at work \xa0is merely a reflection of something else \xa0', ' can breathe easy \xa0after a few moments \xa0', ' world of goodwill and irreverent gaze.     ', 'unky for kids to enjoy , though almost', ' is simply one more way of ragging forward thanks to the three', ' to the direction of the story and the director , who made no major splash', ' .           ', ' how impossible it must be for daggers and headquaters  ', ' pictures performances  \xa0to the machinations \xa0', ' a victimless evil , at least it is that it ipseter', ' and the

## Compute Reward

$\text {reward} = \text {score} - \log (\frac {\pi^{RL}_\theta} {\pi^{SFT}})$

In [ ]:
from copy import deepcopy
sft_model = deepcopy(model)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
input_data = data_collator([
    {'input_ids': ids,
     'attention_mask': torch.ones_like(ids)} for ids in query_response_tensors
]).to(device)
input_data

{'input_ids': tensor([[14150,   925,   257, 11004,  2646,   326, 25409,   290,  5763,   689,
           257,  2092,  1111, 28201,  2612,  9032,   290,   281, 50256, 50256,
         50256, 50256],
        [ 5562,   373,   299,   470,   477,   326,  1049,   837,   475,   340,
          6793,   281,  4296, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256],
        [  292,   262,  5337,   319,   543,   340,   705,    82,  1912, 27452,
         21002, 20589, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256],
        [20797,   262,  2443,   351,   645,  6411,  2181, 43787, 12857, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256],
        [   11,  1748, 17150,   389, 46060,   546,   644,   257,  2646,  2476,
           837,  4737, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256],
        [39563,  4420, 19376,   764,   220,   220,   220,   220, 50256, 5025

In [ ]:
def compute_rewards(input_data, query_tensors, response_tensors, score_tensors):
    with torch.no_grad():
        logits, values = model(**input_data) # b, seq, vocab
        ref_logits, _ = sft_model(**input_data)
        logp = torch.nn.functional.log_softmax(logits[:, :-1, :], dim=-1)
        ref_logp = torch.nn.functional.log_softmax(ref_logits[:, :-1, :], dim=-1)

        labels = input_data['input_ids'][:, 1:] # b, seq

        logp = torch.gather(logp, 2, labels.unsqueeze(-1)).squeeze(-1) # batch, seq
        ref_logp = torch.gather(ref_logp, 2, labels.unsqueeze(-1)).squeeze(-1) # batch, seq

        kl = logp - ref_logp
        beta = 0.2
        rewards = - beta * kl
        attention_mask = input_data['attention_mask']
        masks = torch.zeros_like(attention_mask[:, 1:])
        masks[:,:] = attention_mask[:, 1:]
        for j in range(len(query_tensors)):
            start = len(query_tensors[j]) - 1
            end = start + len(response_tensors[j])
            masks[j, :start] = 0
            masks[j, end:] = 0
            rewards[j, end - 1] += score_tensors[j]
            rewards[j, :] *= masks[j, :]
            values[j, :-1] *= masks[j, :]

    return logp, rewards, values[:, :-1], masks


In [ ]:
logprobs, rewards, values, masks = compute_rewards(input_data, query_tensors, response_tensors, score_tensors)
print(rewards[0])
print(input_data['input_ids'][0])
print(input_data['attention_mask'][0])

tensor([-0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000,
        -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 0.9296, -0.0000,
        -0.0000, -0.0000, -0.0000], device='cuda:0')
tensor([14150,   925,   257, 11004,  2646,   326, 25409,   290,  5763,   689,
          257,  2092,  1111, 28201,  2612,  9032,   290,   281, 50256, 50256,
        50256, 50256], device='cuda:0')
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
       device='cuda:0')


In [ ]:
print(masks[0])
print(values[0])

tensor([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
       device='cuda:0')
tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, 12.4251, 13.3923,
        11.8645, 13.5134, 16.3830, 16.1762, 13.3543, 12.5799,  9.0171,  9.5595,
        11.5133,  0.0000,  0.0000,  0.0000,  0.0000], device='cuda:0')


## Compute Advantage

In [ ]:
def masked_mean(values, mask):
    return (values * mask).sum() / mask.sum()

def masked_var(values, mask):
    mean = masked_mean(values, mask)
    centred_values = values - mean
    return masked_mean(centred_values ** 2, mask)

def masked_whiten(values, mask):
    mean, var = masked_mean(values, mask), masked_var(values, mask)
    whitened = (values - mean) * torch.rsqrt(var + 1e-8)
    whitened += mean
    return whitened

def compute_advantage(rewards, values, masks):
    lastgae = 0.0
    advantage_reversed = []
    seq_length = rewards.shape[-1]
    gamma, lam = 1.0, 0.95

    for t in reversed(range(seq_length)):
        nextvalues = values[:, t + 1] if t < seq_length - 1 else 0.0
        delta = rewards[:, t] + gamma * nextvalues - values[:, t]
        lastgae = delta + gamma * lam * lastgae
        advantage_reversed.append(lastgae)
    advantages = torch.stack(advantage_reversed[::-1], dim=1)
    advantages = masked_whiten(advantages, masks)

    returns = advantages + values
    return advantages, returns


In [ ]:
advantages, returns = compute_advantage(rewards, values, masks)
print(advantages[0])
print(returns[0])

tensor([ -5.3321,  -5.2599,  -5.1839,  -5.1038,  -5.0196,  -4.9310,  -8.6590,
         -9.0594,  -8.7134,  -9.3263, -10.3469, -10.4750,  -9.8056,  -9.7306,
         -8.7942,  -9.0710,  -9.7965,  -6.7042,  -6.7042,  -6.7042,  -6.7042],
       device='cuda:0')
tensor([-5.3321, -5.2599, -5.1839, -5.1038, -5.0196, -4.9310,  3.7661,  4.3329,
         3.1510,  4.1871,  6.0362,  5.7013,  3.5487,  2.8493,  0.2229,  0.4885,
         1.7169, -6.7042, -6.7042, -6.7042, -6.7042], device='cuda:0')


## Mini-batch PPO Training

### Training Config

In [ ]:
learning_rate = 1e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
np.random.permutation(batch_size)

array([ 7, 18, 20, 15, 10, 19, 29, 21,  1, 14, 27,  8, 26, 28, 16,  4, 17,
       13,  9,  2,  6, 22, 12, 11, 24,  5, 23,  0, 25,  3, 31, 30])

In [ ]:
mini_batch_size = 4
ppo_epochs = 4

cliprange_ratio = 0.2

v_loss_coeff = 0.1

ratio_threshold = 10

def compute_loss(old_logprobs, values, logprobs, vpreds, masks, advantages, returns):
    ratio = torch.exp(logprobs - old_logprobs)
    pg_loss1 = - ratio * advantages
    pg_loss2 = - torch.clamp(ratio, 1 - cliprange_ratio, 1 + cliprange_ratio) * advantages
    pg_loss = masked_mean(torch.max(pg_loss1, pg_loss2), masks)

    v_loss = masked_mean((vpreds - returns) ** 2, masks)
    loss = pg_loss + v_loss_coeff * v_loss

    avg_ratio = masked_mean(ratio, masks)
    if avg_ratio > ratio_threshold:
        pg_loss = pg_loss * 0.0
        v_loss = v_loss * 0.0
        loss = loss * 0.0

    return loss, v_loss

def mini_batch_train():
    for ep in range(ppo_epochs):
        batch_inds = np.random.permutation(batch_size)

        for start in range(0, batch_size, mini_batch_size):
            end = start + mini_batch_size
            mini_batch_inds = batch_inds[start:end]

            mb_model_inputs = {
                'input_ids': input_data['input_ids'][mini_batch_inds],
                'attention_mask': input_data['attention_mask'][mini_batch_inds]
            }
            mb_logits, mb_vpreds = model(**mb_model_inputs)
            mb_logits = torch.nn.functional.log_softmax(mb_logits[:, :-1, :], dim=-1)
            mb_logprobs = torch.gather(mb_logits, 2, mb_model_inputs['input_ids'][:, 1:].unsqueeze(-1)).squeeze(-1)

            loss, loss_v = compute_loss(
                logprobs[mini_batch_inds],
                values[mini_batch_inds],
                mb_logprobs,
                mb_vpreds[:, :-1],
                masks[mini_batch_inds],
                advantages[mini_batch_inds],
                returns[mini_batch_inds]
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            print('loss/total', loss.item())
    print('mini-batch training finished')



In [ ]:
mini_batch_train()

loss/total 18.321537017822266
loss/total 18.628482818603516
loss/total 17.12299156188965
loss/total 18.40904998779297
loss/total 17.71307945251465
loss/total 16.77200698852539
loss/total 16.222177505493164
loss/total 16.680198669433594
loss/total 14.740123748779297
loss/total 13.469751358032227
loss/total 14.818939208984375
loss/total 13.319629669189453
loss/total 12.931940078735352
loss/total 12.688895225524902
loss/total 13.255836486816406
loss/total 12.145954132080078
loss/total 11.298210144042969
loss/total 12.194023132324219
loss/total 12.379443168640137
loss/total 11.827146530151367
loss/total 11.281953811645508
loss/total 10.488716125488281
loss/total 10.65282917022705
loss/total 10.22225570678711
loss/total 10.078499794006348
loss/total 10.033658981323242
loss/total 10.205816268920898
loss/total 10.388324737548828
loss/total 9.283001899719238
loss/total 9.927655220031738
loss/total 9.801006317138672
loss/total 9.138940811157227
mini-batch training finished


## Train RLHF

In [ ]:
num_epochs = 1

for epoch in range(num_epochs):
    for batch in train_dataloader:
        # Generate responses
        query_tensors = batch['input_ids']
        query_attention_masks = batch['attention_mask']

        response_tensors = []
        query_response_tensors = []
        score_tensors = []

        for i, query in enumerate(query_tensors):
            query = query.to(device)
            query_attention_mask = query_attention_masks[i].to(device)
            new_tokens = random.choice(list(range(output_min_length, output_max_length)))
            generation_kwargs["max_new_tokens"] = new_tokens
            query_response = model.generate(
                input_ids=query.unsqueeze(0),
                attention_mask=query_attention_mask.unsqueeze(0),
                **generation_kwargs
                ).squeeze(0)

            response_len = len(query_response) - len(query)
            response_tensors.append(query_response[-response_len:])
            query_response_tensors.append(query_response)

            with torch.no_grad():
                query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID]).to(device)])
                attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
                score = reward_model(query_response_score.unsqueeze(0), attention_mask.unsqueeze(0)).squeeze(0)[-1]
                score = 2 * (score - 0.5)
            score_tensors.append(score)

        input_data = data_collator([
            {
                'input_ids': ids,
                'attention_mask': torch.ones_like(ids)
            }
            for ids in query_response_tensors
        ]).to(device)

        # rewards and advantages
        logprobs, rewards, values, masks = compute_rewards(input_data, query_tensors, response_tensors, score_tensors)
        advantages, returns = compute_advantage(rewards, values, masks)

        # mini batch training
        mini_batch_train()
    print(f'epoch {epoch + 1} finished')

loss/total 7.541379928588867
loss/total 7.844137191772461
loss/total 6.611658096313477
loss/total 6.873020172119141
loss/total 7.017009735107422
loss/total 6.217047691345215
loss/total 6.657845497131348
loss/total 6.740623950958252
loss/total 6.418061256408691
loss/total 5.493022918701172
loss/total 5.382020473480225
loss/total 5.56911039352417
loss/total 5.3204803466796875
loss/total 5.080960273742676
loss/total 5.145849227905273
loss/total 5.466140270233154
loss/total 5.569111347198486
loss/total 4.826955795288086
loss/total 4.570194721221924
loss/total 4.5696001052856445
loss/total 4.492641448974609
loss/total 4.899178504943848
loss/total 4.727170944213867
loss/total 4.453793525695801
loss/total 4.60481071472168
loss/total 4.354701995849609
loss/total 4.318787097930908
loss/total 4.062783718109131
loss/total 4.877025604248047
loss/total 4.191400527954102
loss/total 4.612393856048584
loss/total 4.5340576171875
mini-batch training finished
loss/total 3.4324893951416016
loss/total 3.00

KeyboardInterrupt: 

## Validation

In [ ]:
len(tokenized_dataset_val)

807

In [ ]:
val_gen_lengths = [0] * len(tokenized_dataset_val)
for i in range(len(tokenized_dataset_val)):
    val_gen_lengths[i] = random.choice(list(range(output_min_length, output_max_length)))

In [ ]:
val_gen_lengths[:10]

[10, 8, 13, 15, 15, 6, 7, 15, 15, 15]

In [ ]:
def validate():
    scores = []
    for b, batch in enumerate(val_dataloader):
        # Generate_responses
        query_tensors = batch['input_ids']
        query_attention_masks = batch['attention_mask']
        for i, query in enumerate(query_tensors):
            query = query.to(device)
            query_attention_mask = query_attention_masks[i].to(device)
            new_tokens = val_gen_lengths[b * len(query_tensors) + i]
            generation_kwargs["max_new_tokens"] = new_tokens
            query_response = model.generate(
                input_ids=query.unsqueeze(0),
                attention_mask=query_attention_mask.unsqueeze(0),
                **generation_kwargs
                ).squeeze(0)
            query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID]).to(device)])
            attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
            score = reward_model(query_response_score.unsqueeze(0), attention_mask.unsqueeze(0)).squeeze(0)[-1]
            score = 2 * (score - 0.5)
            scores.append(score.item())
    print('avg score:', sum(scores) / len(scores))

In [ ]:
validate()

avg score: 0.11851542900042907


In [ ]:
torch.save(model.state_dict(), 'ppo_model_epoch_1.pt')

In [ ]:
model_path = './sft_model_epoch_1'
model = ModelForCausalLMWithValueHead(model_path).to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
validate()

avg score: 0.06482573763883985
